# Phase 7: train and compare fixed models

This phase makes a first honest comparison using the ten locked, match-grouped folds from Phase 6. It fits four predeclared, untuned models with fresh fold-local preprocessing, then reports pooled out-of-fold predictions. The held-out matches remain untouched; these results are for development and do not show test performance.

## 1. Check the project Python environment

In VS Code, select the project `.venv` kernel before running cells. The check searches the current folder and its parents, so it works when Jupyter starts in the project root or in `notebooks/`.

In [ ]:
from pathlib import Path
import sys

project_root = next((p for p in (Path.cwd(), *Path.cwd().parents) if (p / '.venv').is_dir() and (p / 'README.md').is_file()), None)
if project_root is None:
    raise FileNotFoundError('Could not find the project folder containing both .venv/ and README.md.')
sys.path.insert(0, str(project_root))
expected_prefix = (project_root / '.venv').resolve()
active_prefix = Path(sys.prefix).resolve()
if active_prefix != expected_prefix:
    raise RuntimeError(f'Select the project .venv kernel. Expected {expected_prefix}; active Python is {active_prefix}.')
print(f'Project root: {project_root}')
print(f'Python executable: {sys.executable}')

## 2. Read the fixed model protocol before training

The script verifies the source, holdout, feature contract, and Phase 6 fold map before fitting anything. It writes `phase_07_protocol.json` before the first fit. That file locks four fixed estimators and the 0.5 decision threshold. There is no search for better settings, resampling, class weighting, calibration, or threshold tuning in this phase. If a previous protocol differs from the current locked inputs or settings, the script stops.

## 3. Run cross-validation and fit the final training pipelines

Each model gets a new pipeline in every fold. Scaling and one-hot categories are learned from that fold's training matches only, then applied to its validation matches. The validation map joins by event ID, and all shots from one match stay together. After out-of-fold predictions are complete, each fixed pipeline is refit on all 7,334 training rows and saved for later project phases. Those full-training resubstitution predictions are optimistic and are labeled separately. No held-out test row is used.

In [ ]:
import json
import subprocess

script = project_root / 'scripts' / 'train_models.py'
result = subprocess.run([sys.executable, str(script), '--root', str(project_root)], text=True, capture_output=True)
print(result.stdout)
if result.returncode != 0:
    print(result.stderr)
    raise RuntimeError('Model training failed; read the error above before continuing.')

## 4. Read pooled out-of-fold metrics

Pooled OOF metrics combine each training event's one prediction from the fold where its match was held out. Log loss and Brier score evaluate probability quality; average precision and ROC AUC rank positive cases; accuracy, precision, recall, and F1 use the fixed 0.5 threshold. The Dummy prior model predicts each fold's training goal prevalence. Fold mean and sample standard deviation are also reported, without confidence-interval claims.

In [ ]:
import pandas as pd
from IPython.display import display, Image

with (project_root / 'docs' / 'phase_07_model_report.json').open(encoding='utf-8') as f:
    model_report = json.load(f)
comparison = pd.read_csv(project_root / 'reports' / 'tables' / 'phase07_model_comparison.csv', float_precision='round_trip')
columns = ['model', 'oof_log_loss', 'oof_brier_score', 'oof_average_precision', 'oof_roc_auc', 'oof_accuracy', 'oof_precision', 'oof_recall', 'oof_f1']
display(comparison[columns].sort_values('oof_log_loss'))
print(f"OOF coverage: {model_report['training_shot_count']:,} training events across {model_report['training_match_count']} training matches")
print(f"Threshold rule: predict goal when P(goal) >= {model_report['threshold']}")

## 5. Compare individual folds and full-training resubstitution

The fold table shows each model on each of the ten validation folds. The following four-row summary gives the unweighted fold mean and sample standard deviation for log loss and accuracy; the standard deviation describes variation across these folds and is not a confidence interval.

The final table reports predictions from pipelines refit on all training rows and scored on those same rows. These are training resubstitution results and are optimistic by construction. Use the OOF metrics above for development comparisons; do not treat resubstitution as validation or test performance.

In [ ]:
fold_metrics = pd.read_csv(project_root / 'reports' / 'tables' / 'phase07_fold_metrics.csv', float_precision='round_trip')
fold_columns = ['model', 'fold', 'log_loss', 'brier_score', 'accuracy', 'precision', 'recall']
with pd.option_context('display.max_rows', 40, 'display.max_columns', None, 'display.width', 140):
    display(fold_metrics[fold_columns])

fold_summary_columns = ['model', 'fold_mean_log_loss', 'fold_std_log_loss', 'fold_mean_accuracy', 'fold_std_accuracy']
display(comparison[fold_summary_columns])

resubstitution_columns = ['model', 'training_resubstitution_log_loss', 'training_resubstitution_brier_score', 'training_resubstitution_accuracy', 'training_resubstitution_precision', 'training_resubstitution_recall', 'training_resubstitution_f1']
display(comparison[resubstitution_columns])
print('The last table is an optimistic same-training-row diagnostic, not a validation result.')

## 6. Inspect the training-only figures

The first plot compares pooled OOF log loss, Brier score, and average precision. The confusion plot shows actual rows and predicted columns at the fixed threshold. Neither plot evaluates the test set. A model that leads these untuned development metrics is not automatically the best final model; inspect probability calibration, subgroup behavior, uncertainty, and later locked test results before making portfolio claims.

In [ ]:
display(Image(filename=str(project_root / 'reports' / 'figures' / 'phase07_model_comparison.png')))
display(Image(filename=str(project_root / 'reports' / 'figures' / 'phase07_oof_confusion_matrices.png')))

## Why these model choices need care

KNN compares distances between shots, so standardizing the two geometry columns and adding one-hot category dimensions changes what counts as a nearby shot. Gaussian Naive Bayes assumes conditionally independent features with Gaussian likelihoods; that is only an approximation for binary and one-hot columns. Logistic regression provides a linear probability baseline. The Dummy prior classifier is the prevalence-only benchmark, and its prior is recomputed inside each training fold. None of these fixed settings has been tuned.

The saved full-training pipelines are for later controlled use. Their same-row predictions are resubstitution diagnostics, not validation performance. Do not choose or describe a winner using them. The holdout has not been scored in this phase.

## Saved files

- `docs/phase_07_protocol.json`: immutable model settings and locked input fingerprints.
- `docs/phase_07_model_report.json`: OOF and resubstitution results, output hashes, and scope notes.
- `reports/tables/phase07_fold_metrics.csv`: validation metrics and class counts for each model and fold.
- `reports/tables/phase07_model_comparison.csv`: pooled OOF metrics, unweighted fold summaries, and separately labeled training resubstitution metrics.
- `reports/tables/phase07_oof_predictions.csv` and `phase07_training_predictions.csv`: event-level predictions.
- `models/phase07/`: final pipelines fit on all training rows and their hash manifest.

Save the notebook after running its cells and take screenshots of the comparison table and figures. The next phase can use the saved pipelines while preserving this locked evaluation history.